In [1]:

"""A-5: Bidirectional Associative Memory (BAM) with two vector pairs."""
import numpy as np

In [2]:
def sign_bipolar(v: np.ndarray) -> np.ndarray:
    return np.where(v >= 0, 1, -1)

In [3]:
class BAM:
    def __init__(self, x_patterns: np.ndarray, y_patterns: np.ndarray):
        self.W = np.zeros((x_patterns.shape[1], y_patterns.shape[1]))
        for x, y in zip(x_patterns, y_patterns):
            self.W += np.outer(x, y)

    def recall(self, x_init: np.ndarray, max_iter: int = 10):
        x = x_init.copy()
        y = sign_bipolar(x @ self.W)
        for _ in range(max_iter):
            x_new = sign_bipolar(y @ self.W.T)
            y_new = sign_bipolar(x_new @ self.W)
            if np.array_equal(x_new, x) and np.array_equal(y_new, y):
                break
            x, y = x_new, y_new
        return x, y

In [4]:
def main() -> None:
    # Two training pairs in bipolar form {-1, +1}.
    x_patterns = np.array(
        [
            [1, -1, 1, -1],
            [-1, 1, -1, 1],
        ]
    )
    y_patterns = np.array(
        [
            [1, 1, -1],
            [-1, 1, 1],
        ]
    )

    bam = BAM(x_patterns, y_patterns)
    print("Weight matrix W:\n", bam.W)

    print("\nRecall from clean inputs:")
    correct = 0
    for i, (x, y_true) in enumerate(zip(x_patterns, y_patterns), start=1):
        x_rec, y_rec = bam.recall(x)
        is_ok = np.array_equal(y_rec, y_true)
        correct += int(is_ok)
        print(f"Pair {i}: X={x} -> Y*={y_rec} | Expected={y_true} | Match={is_ok}")

    # Noisy test for first pair.
    noisy_x = np.array([1, -1, -1, -1])
    _, noisy_y_rec = bam.recall(noisy_x)
    print("\nNoisy recall test:")
    print(f"Noisy X={noisy_x} -> Recalled Y={noisy_y_rec}")

    acc = correct / len(x_patterns) * 100
    print(f"\nAssociation accuracy on stored pairs: {acc:.2f}%")

In [5]:
if __name__ == "__main__":
    main()

Weight matrix W:
 [[ 2.  0. -2.]
 [-2.  0.  2.]
 [ 2.  0. -2.]
 [-2.  0.  2.]]

Recall from clean inputs:
Pair 1: X=[ 1 -1  1 -1] -> Y*=[ 1  1 -1] | Expected=[ 1  1 -1] | Match=True
Pair 2: X=[-1  1 -1  1] -> Y*=[-1  1  1] | Expected=[-1  1  1] | Match=True

Noisy recall test:
Noisy X=[ 1 -1 -1 -1] -> Recalled Y=[ 1  1 -1]

Association accuracy on stored pairs: 100.00%
